#### Финальное задание PySpark Холкин Николай

In [1]:
# Библиотеки
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F

from datetime import datetime 

In [2]:
# Создаю SparkSession
spark = SparkSession.builder \
        .appName('Final_task') \
        .master('local[*]') \
        .getOrCreate()

print('Spark сессия открыта', datetime.now()) 

Spark сессия открыта 2026-01-26 12:47:22.354504


##### 1. Загрузка и предварительная обработка данных

In [3]:
# 1.1. Загрузка и вывод схемы: Загрузите файл retail_store_sales.csv.  
# Выведите первые 5 строк загруженного DataFrame и его схему (df.printSchema()).
df_csv_raw = spark.read.csv(
    'data/retail_store_sales.csv',
    header = True,
    inferSchema = True
)

In [131]:
print('1.1 Первые 5 строк')
df_csv_raw.show(5, truncate = True, vertical = True)

1.1 Первые 5 строк
-RECORD 0--------------------------
 Transaction ID   | TXN_6867343    
 Customer ID      | CUST_09        
 Category         | Patisserie     
 Item             | Item_10_PAT    
 Price Per Unit   | 18.5           
 Quantity         | 10.0           
 Total Spent      | 185.0          
 Payment Method   | Digital Wallet 
 Location         | Online         
 Transaction Date | 2024-04-08     
 Discount Applied | true           
-RECORD 1--------------------------
 Transaction ID   | TXN_3731986    
 Customer ID      | CUST_22        
 Category         | Milk Products  
 Item             | Item_17_MILK   
 Price Per Unit   | 29.0           
 Quantity         | 9.0            
 Total Spent      | 261.0          
 Payment Method   | Digital Wallet 
 Location         | Online         
 Transaction Date | 2023-07-23     
 Discount Applied | true           
-RECORD 2--------------------------
 Transaction ID   | TXN_9303719    
 Customer ID      | CUST_02        
 Category

In [132]:
print("1.1. Схема DataFrame:")
df_csv_raw.printSchema()

1.1. Схема DataFrame:
root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Total Spent: double (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: date (nullable = true)
 |-- Discount Applied: boolean (nullable = true)



In [6]:
# 1.2. Очистка названий столбцов: Преобразуйте названия всех столбцов к единому регистру - snake_case.  
# Выведите обновленную схему DataFrame  или названия столбцов, чтобы убедиться в изменении названий.

In [133]:
# Замена пробелов на '_'  И перевод все в нижний регистр нижний регистр
print('1.2 Преобразование название столбцов к snake_case')
new_columns = [col.replace(" ", "_").lower() for col in df_csv_raw.columns]

df_csv_snake_raw = df_csv_raw.toDF(*new_columns)

df_csv_snake_raw.printSchema()

1.2 Преобразование название столбцов к snake_case
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- item: string (nullable = true)
 |-- price_per_unit: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- location: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- discount_applied: boolean (nullable = true)



In [135]:
# 1.3. Преобразование типов данных: Т.к. количесвто товара может быть только целым числом- поменяю его на INT
print("'1.3 Преобразование типа данных Int для столбца 'quantity'")
df_csv_snake_raw_int = df_csv_snake_raw.withColumn(
                            'quantity',
                            F.col('quantity').cast(IntegerType())
)
df_csv_snake_raw_int.printSchema()

'1.3 Преобразование типа данных Int для столбца 'quantity'
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- item: string (nullable = true)
 |-- price_per_unit: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- location: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- discount_applied: boolean (nullable = true)



##### 2. Очистка и валидация данных


In [9]:
# 2.1 Заполнение отсутствующие Price Per Unit
# Если  отсутствует цена за единицу товара , но общая сумма и количество имеются, вычислите цену за единицу и заполните пропущенные значения.
df_csv_fill_per_unit = df_csv_snake_raw_int.withColumn(
    "price_per_unit",
    F.when((F.col('price_per_unit').isNull()
           & F.col('quantity').isNotNull()
           & F.col('total_spent').isNotNull()),
           F.round(F.col('total_spent') / F.col('quantity'),2) )
    .otherwise(F.col('price_per_unit'))
)

In [10]:
# проверка 
print("количество пустых значений колонки 'price_per_unit' до заполнения")
print(df_csv_snake_raw_int.filter(F.col("price_per_unit").isNull()).count())
print("количество пустых значений колонки 'price_per_unit' после заполнения")
print(df_csv_fill_per_unit.filter(F.col("price_per_unit").isNull()).count())

количество пустых значений колонки 'price_per_unit' до заполнения
609
количество пустых значений колонки 'price_per_unit' после заполнения
0


In [13]:
# 2.2. Восстановление отсутствующих Item
#-------------------------------------
# Справочник catalog_items
# отбираю нужные столбцы
# убираю NULL значения
# оставляю только уникальные значения distinct()
catalog_items = df_csv_fill_per_unit \
                .select('category','item','price_per_unit') \
                .filter(
                    F.col('category').isNotNull() 
                    & F.col('item').isNotNull() 
                    & F.col('price_per_unit').isNotNull())\
                .distinct()
 

In [14]:
# Заполнение пустых Item из словаря catalog_items
# Делаю Join со словарем
df_joined = df_csv_fill_per_unit.alias('t').join(
    catalog_items.alias('c'),
    (F.col('t.category') == F.col('c.category')) &
    (F.col('t.price_per_unit') == F.col('c.price_per_unit')),
    'left')


In [28]:
# Заполняю нулевые значений в поле item
df_joined_coalesce = df_joined \
                .select(F.col('t.*'),\
                        F.coalesce(F.col('c.item'), F.col('t.item')) \
                        .alias('item_filled')) \
                        .drop(F.col('t.item')) 

In [30]:
df_joined_coalesce.show(1, truncate = True, vertical = True)

-RECORD 0--------------------------
 transaction_id   | TXN_6867343    
 customer_id      | CUST_09        
 category         | Patisserie     
 price_per_unit   | 18.5           
 quantity         | 10             
 total_spent      | 185.0          
 payment_method   | Digital Wallet 
 location         | Online         
 transaction_date | 2024-04-08     
 discount_applied | true           
 item_filled      | Item_10_PAT    
only showing top 1 row



In [36]:
# 2.3. Заполнение отсутствующих Quantity и Total Spent
#-------------------------------------
print("проверка пропусков 'total_spent', но имеющиеся данные в 'quantity' и 'price_per_unit'")
df_joined_coalesce.filter(F.col('total_spent').isNull()
                          & F.col('quantity').isNotNull()
                          & F.col('price_per_unit').isNotNull()) \
                    .count()

проверка пропусков 'total_spent', но имеющиеся данные в 'quantity' и 'price_per_unit'


0

In [38]:
print("проверка пропусков 'quantity', но имеющиеся данные в 'total_spent' и 'price_per_unit'")
df_joined_coalesce.filter(F.col('quantity').isNull()
                          & F.col('total_spent').isNotNull()
                          & F.col('price_per_unit').isNotNull()) \
                    .count()

проверка пропусков 'quantity', но имеющиеся данные в 'total_spent' и 'price_per_unit'


0

In [43]:
print("Количество всех записей до удаления нулевых значений", df_joined_coalesce.count())

Количество всех записей до удаления нулевых значений 12575


In [52]:
df_full = df_joined_coalesce.filter(F.col('category').isNotNull() 
                                    & F.col('quantity').isNotNull()
                                   & F.col('total_spent').isNotNull()
                                   & F.col('price_per_unit').isNotNull())

In [54]:
print("Количество всех записей после удаления нулевых значений", df_full.count())


Количество всех записей после удаления нулевых значений 11971


##### 3. Разведочный анализ данных

In [115]:
# 3.1. Самые популярные категории товаров:
#-------------------------------------
print('3.1.Топ-5 самых продаваемых категорий товаров по категориям')
popular_category = df_full.groupBy(F.col('category')) \
                    .agg(F.sum('quantity') \
                    .alias('cnt_per_cat')) \
                    .orderBy('cnt_per_cat',  ascending=False) \
                    .show(5, truncate = False)


3.1.Топ-5 самых продаваемых категорий товаров по категориям
+-----------------------------+-----------+
|category                     |cnt_per_cat|
+-----------------------------+-----------+
|Furniture                    |8462       |
|Food                         |8387       |
|Beverages                    |8358       |
|Milk Products                |8339       |
|Electric household essentials|8309       |
+-----------------------------+-----------+
only showing top 5 rows



In [113]:
# 3.2. Анализ среднего чека: 
#-------------------------------------
print('3.2. Среднее значение total_spent для каждого метода оплаты')
avg_total_spent_by_payment_method = df_full.groupBy(F.col('payment_method'))\
                                    .agg(F.round(F.avg('total_spent'),2).alias('avg_spent'))\
                                    .show()

3.2. Среднее значение total_spent для каждого метода оплаты
+--------------+---------+
|payment_method|avg_spent|
+--------------+---------+
|   Credit Card|   129.13|
|Digital Wallet|   128.72|
|          Cash|   131.05|
+--------------+---------+



In [114]:
print('3.2 Среднее значение total_spent для каждого меcта оплаты')
avg_total_spent_by_location = df_full.groupBy(F.col('location'))\
                                    .agg(F.round(F.avg('total_spent'),2).alias('avg_spent'))\
                                    .show()

3.2 Среднее значение total_spent для каждого меcта оплаты
+--------+---------+
|location|avg_spent|
+--------+---------+
|In-store|   128.86|
|  Online|   130.42|
+--------+---------+



##### 4. Генерация признаков 

In [106]:
# 4.1. Временные признаки: на основе Transaction Date добавить столбцы day_of_week и transaction_month
# Число (0=Понедельник, 6=Воскресенье)
df_full_with_week_month = df_full \
                        .withColumn('day_of_week', F.weekday('transaction_date'))\
                        .withColumn('transaction_month', F.date_format('transaction_date', 'MMMM'))

In [112]:
print('4.1. Пример df с временными признаками')
df_full_with_week_month.show(1, vertical = True)

4.1. Пример df с временными признаками
-RECORD 0---------------------------
 transaction_id    | TXN_6867343    
 customer_id       | CUST_09        
 category          | Patisserie     
 price_per_unit    | 18.5           
 quantity          | 10             
 total_spent       | 185.0          
 payment_method    | Digital Wallet 
 location          | Online         
 transaction_date  | 2024-04-08     
 discount_applied  | true           
 item_filled       | Item_10_PAT    
 day_of_week       | 0              
 transaction_month | April          
only showing top 1 row



In [109]:
print('4.2. Среднее значение total_spent для каждого дня недели')
avg_total_spent_by_days_of_week = df_full_with_week_month.groupBy(F.col('day_of_week'))\
                                    .agg(F.round(F.avg('total_spent'),2).alias('avg_spent'))\
                                    .orderBy('day_of_week')\
                                    .show()

Среднее значение total_spent для каждого дня недели
+-----------+---------+
|day_of_week|avg_spent|
+-----------+---------+
|          0|   125.57|
|          1|   129.51|
|          2|   126.82|
|          3|   129.28|
|          4|   134.64|
|          5|   131.49|
|          6|   130.18|
+-----------+---------+



In [111]:
print('4.3.Среднее значение total_spent для каждого месяца')
vg_total_spent_by_months = df_full_with_week_month.groupBy(F.col('transaction_month'))\
                                    .agg(F.round(F.avg('total_spent'),2).alias('avg_spent'))\
                                    .orderBy('transaction_month')\
                                    .show()

Среднее значение total_spent для каждого месяца
+-----------------+---------+
|transaction_month|avg_spent|
+-----------------+---------+
|            April|   131.81|
|           August|   124.28|
|         December|   133.15|
|         February|   130.66|
|          January|   134.69|
|             July|   126.57|
|             June|   130.95|
|            March|   126.83|
|              May|    127.4|
|         November|   128.79|
|          October|   127.85|
|        September|   131.45|
+-----------------+---------+



In [136]:
# 4.4. Признаки клиента: Рассчитайте customer_lifetime_value (CLV) 
print('4.4 топ-10 клиентов по CLV')
clv = df_full\
        .groupBy(F.col('customer_id')) \
        .agg(F.sum('total_spent').alias('CLV'))\
        .orderBy('CLV', ascending = False) \
        .show(10)

4.4 топ-10 клиентов по CLV
+-----------+-------+
|customer_id|    CLV|
+-----------+-------+
|    CUST_24|68452.0|
|    CUST_08|67351.5|
|    CUST_05|66974.5|
|    CUST_16|65570.5|
|    CUST_13|65037.0|
|    CUST_23|64507.0|
|    CUST_10|63155.5|
|    CUST_15|63117.5|
|    CUST_21|62933.0|
|    CUST_02|62046.5|
+-----------+-------+
only showing top 10 rows



In [138]:
spark.stop()
print('Сессия Spark отсановлена', datetime.now())

Сессия Spark отсановлена 2026-01-26 14:50:54.373117
